# 第7周 · 课堂练习：纸上反向传播与"梯度覆盖"事故

四个小练习：先在纸上走一遍反向传播；给 `Value` 补两个运算；写数值梯度当"阳性对照"；最后亲手重演（并解释）micrograd 视频里那个著名的梯度 bug。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import math

from zero2researcher.checks import week07 as ck
from zero2researcher.checks import report

## E1 · 纸上反向传播

表达式还是课上的形状：$L = (a \cdot b + c)^2$，但换一组数：$a=3,\ b=-1,\ c=5$。

**全程不写代码**：先前向算出 $e = a \cdot b$、$d = e + c$、$L$；再从 $L$ 往回推每个梯度（提示：$\partial L/\partial d = 2d$；加法原样放行；乘法乘上另一个因子的值）；最后把三个数填进下面的变量。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
ck.paper_backprop(grad_a, grad_b, grad_c)

## E2 · 给 Value 补上取负与减法

下面是课上写完的 `Value`（加法、乘法、tanh、backward 都已就位）。补上两个新方法——**都不需要新的导数公式**，用已有运算拼出来即可，反向传播自动顺着已有运算走：

- `__neg__`：写 `-v` 时被调用，返回 `self * -1`；
- `__sub__`：写 `a - b` 时被调用，返回 `self + (-other)`；
- `__rsub__`（已写好）：处理 `1.0 - v` 这种"左边是普通数字"的情况。

In [ ]:
class Value:
    """课上写完的标量自动微分节点 + 你要补的两个运算"""

    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):        # 1.0 + v
        return self + other

    def __rmul__(self, other):        # 2 * v
        return self * other

    def __neg__(self):                # -v
        # 在这里写你的代码
        raise NotImplementedError

    def __sub__(self, other):         # a - b
        # 在这里写你的代码
        raise NotImplementedError

    def __rsub__(self, other):        # 1.0 - v（已写好：注意方向，是 other 减 self）
        return other + (-self)

    def backward(self):
        topo, visited = [], set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


a, b = Value(5.0), Value(3.0)
y = (a - b) + (1.0 - b)
y.backward()
print(f"y = {y.data}   a.grad = {a.grad}   b.grad = {b.grad}")
ck.value_sub(Value)

## E3 · 数值梯度：阳性对照随身带

把课上的 `numerical_grad(f, x, h=1e-5)`（中心差分）再默写一遍——以后每实现一个新运算，它都是第一道检查。

然后用它检查一个表达式：$f(x) = \tanh(x^2)$ 在 $x = 0.8$ 处。解析侧用你的 `Value`（$x^2$ 先写成 `x * x`——E2 的类还没有乘方），数值侧用 `math.tanh`，相对误差应小于 $10^{-6}$。

In [ ]:
def numerical_grad(f, x, h=1e-5):
    # 在这里写你的代码
    raise NotImplementedError


def rel_error(g1, g2):
    return abs(g1 - g2) / max(abs(g1), abs(g2), 1e-12)


ck.numerical_grad(numerical_grad)

# 解析梯度（Value 的 backward）vs 数值梯度（中心差分），同一个 f(x) = tanh(x²)
# 在这里写你的代码
raise NotImplementedError
print(f"解析 {analytic:.10f}   数值 {numeric:.10f}")
report("tanh(x²) 的解析 vs 数值梯度", rel_error(analytic, numeric) < 1e-6,
       f"相对误差 {rel_error(analytic, numeric):.2e}，应 < 1e-6",
       "检查 backward 是否被调用过；tanh 的局部导数是 1 - t²")

## E4 · 故障重演：梯度被覆盖

micrograd 视频在 1:22:28 专门修过一个 bug，现在你亲手重演它。下面的 `BuggyValue` 和正确实现只差两个字符：`_backward` 里写了 `=` 而不是 `+=`。

考题是最小的复用场景：$b = a + a$。显然 $b = 2a$，所以 $\partial b/\partial a = 2$。看看 bug 版算出多少：

In [ ]:
class BuggyValue:
    """只有加法的最小 Value——但 _backward 用了 =（bug！）"""

    def __init__(self, data, _children=()):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __add__(self, other):
        out = BuggyValue(self.data + other.data, (self, other))

        def _backward():
            self.grad = out.grad      # ← 应该是 +=
            other.grad = out.grad     # ← 应该是 +=
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


a = BuggyValue(3.0)
b = a + a
b.backward()
print("bug 版：b = a + a 反向后 a.grad =", a.grad, "  （数学上应为 2）")

In [ ]:
# 同一个自查函数测 bug 版：它应当报红（❌）——这一格"故意失败"
ck.gradient_reuse(BuggyValue)

**问题**：为什么错的结果是 1，而不是别的数？用"两条路径各传一份梯度"的语言，写两三句话解释 `=` 把什么覆盖掉了。再想一步：如果表达式是 $b = a + a + a$，bug 版会给出多少？（答案仍是 1——想想为什么。）

（在这里回答）

In [ ]:
# 用你 E2 的（正确的）Value 验证：同一个场景 a.grad 应为 2，这次应当是 ✅
ck.gradient_reuse(Value)